In [6]:
"""
Pipeline: LIF -> Cellpose + Ilastik -> per-cell stats (no per-cell TIFFs)

Requirements (conda env e.g. `cellpose3-gpu-python`):
    pip install readlif tifffile numpy scikit-image cellpose pandas

Optional (for Ilastik headless):
    - ilastik installed
    - ILASTIK_EXE path set below
"""

import os
import pathlib
import subprocess
from typing import Dict, List

import numpy as np
import pandas as pd
import tifffile as tiff
from skimage.segmentation import find_boundaries
from cellpose import models
from readlif.reader import LifFile

# ==========================
# CONFIG
# ==========================

# --- paths ---
OUTPUT_DIR = r"C:\Users\JackM\5C_signalling\caspase\masks_python"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Subdirs for different outputs
ORIG_DIR     = os.path.join(OUTPUT_DIR, "original_full_images")
FULLMASK_DIR = os.path.join(OUTPUT_DIR, "full_masks")
QC_DIR       = os.path.join(OUTPUT_DIR, "QC")

for _d in [ORIG_DIR, FULLMASK_DIR, QC_DIR]:
    os.makedirs(_d, exist_ok=True)

# Path to ilastik executable (for headless prediction)
ILASTIK_EXE = r"C:\Program Files\ilastik-1.4.0.post1\ilastik.exe"  # <- change to your install

if not os.path.exists(ILASTIK_EXE):
    raise FileNotFoundError(f"ILASTIK_EXE does not exist: {ILASTIK_EXE}")

# Cellpose custom model paths
NUC_MODEL_PATH   = r"C:\Users\JackM\5C_signalling\caspase\10xNuclei\models\10xNuc3"
CELL_MODEL_PATH  = r"C:\Users\JackM\5C_signalling\caspase\cellpose\models\10x_OPCs_14"

# LIF files + associated Ilastik projects
LIF_CONFIG: Dict[str, Dict[str, str]] = {
    # --------- N1 ---------
    #r"C:\Users\JackM\5C_signalling\caspase\N1_0922_caspase_real.lif": {
    #    "mbp_ilp":     r"C:\Users\JackM\5C_signalling\caspase\N1_MBP\N1_MBP.ilp",
    #    "pdgfra_ilp":  r"C:\Users\JackM\5C_signalling\caspase\N1_PDGFRa\N1_pdgfra.ilp",
    #    "o4_ilp":      r"C:\Users\JackM\5C_signalling\caspase\N1_O4\N1_O4.ilp",
    #    "casp_ilp":    r"C:\Users\JackM\5C_signalling\caspase\N1_caspase\N1_caspase.ilp",
    #},
    # --------- N4 ---------
    #r"D:\caspase\N4_0929_caspase.lif": {
    #    "mbp_ilp":     r"D:\caspase\N4_MBP\N4_mbp.ilp",
    #    "pdgfra_ilp":  r"D:\caspase\N4_PDGFRa\N4_pdgra.ilp",
    #    "o4_ilp":      r"D:\caspase\N4_O4\N4_O4.ilp",
    #    "casp_ilp":    r"D:\caspase\N4_caspase\N4_caspase.ilp",
    #},
    # --------- N6 ---------
    #r"D:\caspase\N6_1013_Caspase.lif": {
    #    "mbp_ilp":     r"D:\caspase\N6_MBP\N6_mbp.ilp",
    #    "pdgfra_ilp":  r"D:\caspase\N6_PDGFRa\N6_pdgfra.ilp",
    #    "o4_ilp":      r"D:\caspase\N6_O4\N6_O4.ilp",
    #    "casp_ilp":    r"C:\Users\JackM\N6_caspase.ilp",
    #},
    # --------- N7 ---------
    #r"D:\caspase\N7_20251020_caspase.lif": {
     #   "mbp_ilp":     r"D:\caspase\N7_MBP\N7_mbp.ilp",
     #   "pdgfra_ilp":  r"D:\caspase\N7_PDGFRa\N7_pdgfra.ilp",
     #   "o4_ilp":      r"D:\caspase\N7_O4\N7_O4.ilp",
     #   "casp_ilp":    r"D:\caspase\N7_caspase\N7_caspase.ilp",
    #},
       # --------- N9 ---------
    r"C:\Users\JackM\5C_signalling\caspase\N9_20251215_caspase.lif": {
        "mbp_ilp":     r"C:\Users\JackM\5C_signalling\caspase\N9_MBP\\N9_mbp.ilp",
        "pdgfra_ilp":  r"C:\Users\JackM\5C_signalling\caspase\N9_PDGFRa\\n9_pdgfra.ilp",
        "o4_ilp":      r"C:\Users\JackM\5C_signalling\caspase\N9_O4\\n9_O4.ilp",
        "casp_ilp":    r"C:\Users\JackM\5C_signalling\caspase\N9_caspase\\n9_caspase.ilp",
    }


}

# Channel indices (0-based) for data read as (C, Y, X)
# Macro mapping was: 1=DAPI, 2=PDGFRa, 3=O4, 4=Caspase, 5=MBP
CHAN_DAPI     = 0
CHAN_PDGFRa   = 1
CHAN_O4       = 2
CHAN_CASPASE  = 3
CHAN_MBP      = 4

# Cellpose parameters
NUC_DIAMETER   = 15.0
CELL_DIAMETER  = 30.0


# ==========================
# HELPERS
# ==========================

def run_ilastik_on_array(
    img: np.ndarray,
    ilp_path: str,
    tmp_root: pathlib.Path,
    tag: str
) -> np.ndarray:
    """
    Run Ilastik pixel classification 'Simple Segmentation' on a 2D image array.

    We:
      - delete old files for this tag in tmp_root
      - write tmp_root/<tag>_in.tiff as float64
      - tell Ilastik to write tmp_root/<tag>_seg.tiff
      - read exactly that file and return as uint8

    This matches the actual filenames you see, e.g.
    N1_0922_caspase_real_S1_MDL29951_9_R5_MBP_seg.tiff
    """
    tmp_root.mkdir(parents=True, exist_ok=True)

    # Clean any old junk for this tag (.tif and .tiff)
    for p in tmp_root.glob(f"{tag}_*.tif*"):
        try:
            p.unlink()
        except Exception:
            pass

    # Full paths for input and expected output
    tmp_in  = tmp_root / f"{tag}_in.tiff"
    tmp_out = tmp_root / f"{tag}_seg.tiff"   # NOTE: .tiff (two f’s)

    # Write input for Ilastik as float64 (matches your 64-bit training)
    tiff.imwrite(tmp_in, img.astype(np.float64))

    cmd = [
        ILASTIK_EXE,
        "--headless",
        f"--project={ilp_path}",
        "--export_source=Simple Segmentation",
        "--output_format=tiff",
        f"--output_filename_format={tmp_out}",  # full path, not just a name
        str(tmp_in),
    ]

    print(f"[ilastik] running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Ilastik failed for tag '{tag}'.\n"
            f"Command:\n{' '.join(cmd)}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    # Now require that exact seg file to exist
    if not tmp_out.exists():
        existing = [p.name for p in tmp_root.glob("*.tif*")]
        raise FileNotFoundError(
            f"Expected Ilastik output '{tmp_out}' not found for tag '{tag}'.\n"
            f"Existing TIFFs in {tmp_root}:\n{existing}\n\n"
            f"Command was:\n{' '.join(cmd)}\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    print(f"[ilastik] using output: {tmp_out}")

    seg = tiff.imread(tmp_out)

    # Clean up input; keep seg so you can open it in Fiji if you want
    try:
        tmp_in.unlink(missing_ok=True)
        # If you want to delete the seg too, uncomment this:
        # tmp_out.unlink(missing_ok=True)
    except Exception:
        pass

    if seg.ndim > 2:
        seg = np.squeeze(seg)

    return seg.astype(np.uint8)


def binary_reconstruct(mask: np.ndarray, seed: np.ndarray) -> np.ndarray:
    """
    Binary reconstruction: keep only regions of 'mask' connected to 'seed'.
    seed and mask are boolean arrays.
    """
    from skimage.morphology import reconstruction as sk_reconstruction

    seed = seed & mask
    rec = sk_reconstruction(
        seed.astype(np.uint8),
        mask.astype(np.uint8),
        method='dilation'
    )
    return (rec > 0).astype(np.uint8)


def build_final_mask(
    dapi_mask: np.ndarray,
    mbp_seg: np.ndarray,
    pdgfra_seg: np.ndarray,
    o4_seg: np.ndarray,
    casp_seg: np.ndarray
) -> np.ndarray:
    """
    Stack 5 binary masks into a 5xYxX uint8 array.
    Order: [DAPI, MBP, PDGFRa, O4, Caspase]
    """
    stack = np.stack([
        dapi_mask,
        mbp_seg,
        pdgfra_seg,
        o4_seg,
        casp_seg,
    ], axis=0).astype(np.uint8)

    return stack


def load_scene_as_cyx_and_name(lif: LifFile, scene_idx: int):
    """
    Use readlif to load a single scene as a (C, Y, X) float64 array
    AND return the raw series name from the LIF.

    No autoscaling: preserve raw intensities from the LIF.
    """
    lif_img = lif.get_image(scene_idx)
    raw_name = str(lif_img.name)  # e.g. "Series 16: RWT9996/10µM"

    chans = []
    for ch_img in lif_img.get_iter_c(z=0, t=0, m=0):
        chans.append(np.array(ch_img))

    data = np.stack(chans, axis=0).astype(np.float64)  # (C, Y, X)

    return data, raw_name


def make_qc_overlay(
    base_name: str,
    dapi_mask: np.ndarray,
    mbp_mask: np.ndarray,
    pdgfra_mask: np.ndarray,
    o4_mask: np.ndarray,
    casp_mask: np.ndarray,
    cell_labels: np.ndarray,
    out_dir: str
):
    """
    Build an RGB QC image:
      - different colours per stain (full image)
      - Cellpose cell boundaries overlaid in white
    Save as {base_name}_QC.tif in out_dir.
    """
    from skimage.segmentation import find_boundaries

    Y, X = dapi_mask.shape
    rgb = np.zeros((Y, X, 3), dtype=np.float32)

    # Base intensities (0.0-1.0)
    d = dapi_mask.astype(np.float32)
    m = mbp_mask.astype(np.float32)
    p = pdgfra_mask.astype(np.float32)
    o = o4_mask.astype(np.float32)
    c = casp_mask.astype(np.float32)

    # Colour mapping:
    # DAPI      -> blue
    # MBP       -> green
    # PDGFRa    -> red
    # O4        -> cyan
    # Caspase   -> yellow

    rgb[..., 2] += d * 0.6                      # blue: DAPI
    rgb[..., 1] += m * 0.7                      # green: MBP
    rgb[..., 0] += p * 0.7                      # red: PDGFRa
    rgb[..., 1] += o * 0.5; rgb[..., 2] += o*0.5  # cyan: O4
    rgb[..., 0] += c * 0.5; rgb[..., 1] += c*0.5  # yellow: Caspase

    # Cellpose boundaries in white
    if cell_labels is not None:
        bounds = find_boundaries(cell_labels, mode="inner")
        rgb[bounds, 0] = 1.0
        rgb[bounds, 1] = 1.0
        rgb[bounds, 2] = 1.0

    rgb = np.clip(rgb, 0, 1) * 255.0
    rgb_uint8 = rgb.astype(np.uint8)

    out_path = pathlib.Path(out_dir) / f"{base_name}_QC.tif"
    tiff.imwrite(out_path, rgb_uint8, photometric="rgb")
    print(f"[QC] wrote {out_path}")


# ==========================
# MAIN PROCESSING
# ==========================

def process_lif(
    lif_path: str,
    cfg: Dict[str, str],
    nuc_model,
    cell_model,
    stats_records: List[Dict]
):
    lif_path = pathlib.Path(lif_path)
    print(f"\n=== Processing LIF: {lif_path} ===")

    lif = LifFile(str(lif_path))
    num_scenes = lif.num_images

    tmp_root = pathlib.Path(OUTPUT_DIR) / "_tmp"
    tmp_root.mkdir(parents=True, exist_ok=True)

    for scene_idx in range(num_scenes):
        # Data as (C, Y, X) and raw series name
        data, raw_series_name = load_scene_as_cyx_and_name(lif, scene_idx)
        C, Y, X = data.shape
        print(f"  Scene {scene_idx} / {num_scenes-1}: "
              f"name='{raw_series_name}', shape CYX={data.shape}")

        # ---- base_name that keeps treatment from series name ----
        parts = raw_series_name.split(":", 1)
        if len(parts) > 1:
            series_desc = parts[1].strip()
        else:
            series_desc = raw_series_name.strip()

        series_desc_clean = (
            series_desc.replace(" ", "_")
                       .replace("/", "_")
                       .replace("\\", "_")
                       .replace("µ", "u")
                       .replace("μ", "u")
                       .replace(":", "_")
        )

        base_name = (
            f"{lif_path.stem}_S{scene_idx+1}_{series_desc_clean}"
            .replace("-", "_")
            .replace("+", "_")
        )

        # ---------------------------
        # 0. Save original full image as TIFF (for viewing in Fiji)
        # ---------------------------
        orig_path = pathlib.Path(ORIG_DIR) / f"{base_name}_orig.tif"
        tiff.imwrite(
            orig_path,
            data.astype(np.float32),
            imagej=True,
            metadata={"axes": "CYX"}
        )
        print(f"[orig] wrote {orig_path}")

        # ---------------------------
        # 1. Nuclei: Cellpose on DAPI
        # ---------------------------
        dapi = data[CHAN_DAPI]

        nuc_res = nuc_model.eval(
            dapi,
            diameter=NUC_DIAMETER,
            channels=[0, 0],
            do_3D=False
        )
        masks_nuc = nuc_res[0]

        dapi_mask = (masks_nuc > 0).astype(np.uint8)

        # Save DAPI mask seed for debugging
        tiff.imwrite(
            pathlib.Path(QC_DIR) / f"{base_name}_DAPI_mask.tif",
            dapi_mask.astype(np.uint8),
            imagej=True
        )

        # ---------------------------
        # 2–6. Ilastik channels + cellpose + stats
        # If anything in here fails (e.g. Ilastik on a single image),
        # flag it and skip to the next scene.
        # ---------------------------
        try:
            # 2. Marker segmentations via Ilastik
            mbp_raw     = data[CHAN_MBP]
            pdgfra_raw  = data[CHAN_PDGFRa]
            o4_raw      = data[CHAN_O4]
            casp_raw    = data[CHAN_CASPASE]

            # --- MBP ---
            mbp_seg_raw = run_ilastik_on_array(
                mbp_raw, cfg["mbp_ilp"], tmp_root, f"{base_name}_MBP"
            )
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_MBP_seg_raw.tif",
                mbp_seg_raw.astype(np.uint8),
                imagej=True
            )
            mbp_mask_raw = (mbp_seg_raw == 1)  # 1 = positive class
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_MBP_mask_raw.tif",
                (mbp_mask_raw.astype(np.uint8) * 255),
                imagej=True
            )
            mbp_clean = binary_reconstruct(mbp_mask_raw, dapi_mask > 0)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_MBP_mask_clean.tif",
                (mbp_clean.astype(np.uint8) * 255),
                imagej=True
            )

            # --- PDGFRa ---
            pdgfra_seg_raw = run_ilastik_on_array(
                pdgfra_raw, cfg["pdgfra_ilp"], tmp_root, f"{base_name}_PDGF"
            )
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_PDGF_seg_raw.tif",
                pdgfra_seg_raw.astype(np.uint8),
                imagej=True
            )
            pdgfra_mask_raw = (pdgfra_seg_raw == 1)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_PDGF_mask_raw.tif",
                (pdgfra_mask_raw.astype(np.uint8) * 255),
                imagej=True
            )
            pdgfra_clean = binary_reconstruct(pdgfra_mask_raw, dapi_mask > 0)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_PDGF_mask_clean.tif",
                (pdgfra_clean.astype(np.uint8) * 255),
                imagej=True
            )

            # --- O4 ---
            o4_seg_raw = run_ilastik_on_array(
                o4_raw, cfg["o4_ilp"], tmp_root, f"{base_name}_O4"
            )
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_O4_seg_raw.tif",
                o4_seg_raw.astype(np.uint8),
                imagej=True
            )
            o4_mask_raw = (o4_seg_raw == 1)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_O4_mask_raw.tif",
                (o4_mask_raw.astype(np.uint8) * 255),
                imagej=True
            )
            o4_clean = binary_reconstruct(o4_mask_raw, dapi_mask > 0)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_O4_mask_clean.tif",
                (o4_clean.astype(np.uint8) * 255),
                imagej=True
            )

            # --- Caspase ---
            casp_seg_raw = run_ilastik_on_array(
                casp_raw, cfg["casp_ilp"], tmp_root, f"{base_name}_CASP"
            )
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_CASP_seg_raw.tif",
                casp_seg_raw.astype(np.uint8),
                imagej=True
            )
            casp_mask_raw = (casp_seg_raw == 1)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_CASP_mask_raw.tif",
                (casp_mask_raw.astype(np.uint8) * 255),
                imagej=True
            )
            casp_clean = binary_reconstruct(casp_mask_raw, dapi_mask > 0)
            tiff.imwrite(
                pathlib.Path(QC_DIR) / f"{base_name}_CASP_mask_clean.tif",
                (casp_clean.astype(np.uint8) * 255),
                imagej=True
            )

            # 3. Build final multichannel mask (5 x Y x X)
            final_mask = build_final_mask(
                dapi_mask=dapi_mask,
                mbp_seg=mbp_clean,
                pdgfra_seg=pdgfra_clean,
                o4_seg=o4_clean,
                casp_seg=casp_clean
            )

            # 3a. Save each full-image mask as its own TIFF (final masks)
            tiff.imwrite(
                pathlib.Path(FULLMASK_DIR) / f"{base_name}_mask_DAPI.tif",
                dapi_mask.astype(np.uint8),
                imagej=True
            )
            tiff.imwrite(
                pathlib.Path(FULLMASK_DIR) / f"{base_name}_mask_MBP.tif",
                mbp_clean.astype(np.uint8),
                imagej=True
            )
            tiff.imwrite(
                pathlib.Path(FULLMASK_DIR) / f"{base_name}_mask_PDGFRa.tif",
                pdgfra_clean.astype(np.uint8),
                imagej=True
            )
            tiff.imwrite(
                pathlib.Path(FULLMASK_DIR) / f"{base_name}_mask_O4.tif",
                o4_clean.astype(np.uint8),
                imagej=True
            )
            tiff.imwrite(
                pathlib.Path(FULLMASK_DIR) / f"{base_name}_mask_Caspase.tif",
                casp_clean.astype(np.uint8),
                imagej=True
            )

            # 4. Cellpose on composite to get cell labels
            c1 = data[CHAN_DAPI]
            c2 = data[CHAN_PDGFRa]
            c3 = data[CHAN_O4]
            c5 = data[CHAN_MBP]

            comp_sum = c5 + c3 + c2
            comp_img = np.stack([c1, comp_sum], axis=-1)

            cell_res = cell_model.eval(
                comp_img,
                diameter=CELL_DIAMETER,
                channels=[2, 1],
                do_3D=False
            )
            masks_cell = cell_res[0]

            max_label = int(masks_cell.max())
            if max_label < 65535:
                labels_out = masks_cell.astype(np.uint16)
            else:
                labels_out = masks_cell.astype(np.uint32)

            tiff.imwrite(
                pathlib.Path(FULLMASK_DIR) / f"{base_name}_cellpose_labels.tif",
                labels_out,
                imagej=True
            )

            # 5. QC composite: coloured masks + cellpose outline (full image)
            make_qc_overlay(
                base_name=base_name,
                dapi_mask=dapi_mask,
                mbp_mask=mbp_clean,
                pdgfra_mask=pdgfra_clean,
                o4_mask=o4_clean,
                casp_mask=casp_clean,
                cell_labels=masks_cell,
                out_dir=QC_DIR,
            )

            # 6. Per-cell stats instead of per-cell TIFFs
            labels = np.unique(masks_cell)
            labels = labels[labels > 0]

            dapi_b   = dapi_mask.astype(bool)
            mbp_b    = mbp_clean.astype(bool)
            pdgfra_b = pdgfra_clean.astype(bool)
            o4_b     = o4_clean.astype(bool)
            casp_b   = casp_clean.astype(bool)

            for cid in labels:
                cell_b = (masks_cell == cid)

                cell_area = int(np.count_nonzero(cell_b))
                if cell_area == 0:
                    continue

                dapi_area   = int(np.count_nonzero(dapi_b   & cell_b))
                mbp_area    = int(np.count_nonzero(mbp_b    & cell_b))
                pdgfra_area = int(np.count_nonzero(pdgfra_b & cell_b))
                o4_area     = int(np.count_nonzero(o4_b     & cell_b))
                casp_area   = int(np.count_nonzero(casp_b   & cell_b))

                stats_records.append({
                    "lif_path":   str(lif_path),
                    "lif_name":   lif_path.stem,
                    "scene_idx":  scene_idx,
                    "series_name": raw_series_name,
                    "base_name":  base_name,
                    "cell_id":    int(cid),
                    "cell_area_px":      cell_area,
                    "dapi_area_px":      dapi_area,
                    "mbp_area_px":       mbp_area,
                    "pdgfra_area_px":    pdgfra_area,
                    "o4_area_px":        o4_area,
                    "caspase_area_px":   casp_area,
                })

        except Exception as e:
            # Flag failed scene and move on
            print(
                f"[ilastik ERROR] Skipping scene due to failure in LIF '{lif_path.name}', "
                f"scene {scene_idx}, base_name '{base_name}'.\n"
                f"               Reason: {e}"
            )
            # (Optional) append to a simple log file
            try:
                log_path = pathlib.Path(QC_DIR) / "ilastik_failures.log"
                with open(log_path, "a", encoding="utf-8") as f:
                    f.write(
                        f"{lif_path.name}\tScene {scene_idx}\t{base_name}\t{repr(e)}\n"
                    )
            except Exception:
                pass
            continue  # go to next scene


def main():
    print("[cellpose] loading models...")
    nuc_model  = models.CellposeModel(pretrained_model=NUC_MODEL_PATH,  gpu=True)
    cell_model = models.CellposeModel(pretrained_model=CELL_MODEL_PATH, gpu=True)

    stats_records: List[Dict] = []

    for lif_path, cfg in LIF_CONFIG.items():
        process_lif(lif_path, cfg, nuc_model, cell_model, stats_records)

    if stats_records:
        df = pd.DataFrame(stats_records)
        out_csv = os.path.join(OUTPUT_DIR, "cell_stain_area_stats_N3.csv")
        df.to_csv(out_csv, index=False)
        print(f"[stats] wrote {out_csv}")
    else:
        print("[stats] no cells found; nothing written.")


if __name__ == "__main__":
    main()


[cellpose] loading models...

=== Processing LIF: C:\Users\JackM\5C_signalling\caspase\N9_20251215_caspase.lif ===
  Scene 0 / 119: name='clemastine/9/R1', shape CYX=(5, 2048, 2048)
[orig] wrote C:\Users\JackM\5C_signalling\caspase\masks_python\original_full_images\N9_20251215_caspase_S1_clemastine_9_R1_orig.tif
[ilastik] running: C:\Program Files\ilastik-1.4.0.post1\ilastik.exe --headless --project=C:\Users\JackM\5C_signalling\caspase\N9_MBP\\N9_mbp.ilp --export_source=Simple Segmentation --output_format=tiff --output_filename_format=C:\Users\JackM\5C_signalling\caspase\masks_python\_tmp\N9_20251215_caspase_S1_clemastine_9_R1_MBP_seg.tiff C:\Users\JackM\5C_signalling\caspase\masks_python\_tmp\N9_20251215_caspase_S1_clemastine_9_R1_MBP_in.tiff
[ilastik] using output: C:\Users\JackM\5C_signalling\caspase\masks_python\_tmp\N9_20251215_caspase_S1_clemastine_9_R1_MBP_seg.tiff
[ilastik] running: C:\Program Files\ilastik-1.4.0.post1\ilastik.exe --headless --project=C:\Users\JackM\5C_signalli